# 🏆 M1MLOP — Jour 5 : Intégration Complète & Projet Final
**École IT — Master 1 — Unité 3 BigData — Semaine 11**  
**Amadou MAIGA**

---

## 🗂️ Table des matières
1. [Objectifs et architecture complète](#1)
2. [Dataset Tickets Support](#2)
3. [TP 5.1 — Pipeline Airflow NLP complet](#3)
4. [TP 5.2 — API FastAPI avec monitoring Prometheus](#4)
5. [TP 5.3 — Optimisation : Batching & Caching](#5)
6. [TP 5.4 — Kubernetes HPA + Rolling Update](#6)
7. [🎯 PROJET FINAL — Ticket Classifier System](#7)
8. [Tests automatisés du projet](#8)
9. [Résumé de la formation & bilan final](#9)

---

> 📌 **Prérequis** : Jours 1-4 complétés  
> ⏱️ **Durée estimée** : 7h (matinée + après-midi)  
> 🎯 **Livrable** : Pipeline MLOps complet + Ticket Classifier

<a id='1'></a>
## 1. 🎯 Objectifs du Jour 5

À la fin de cette journée, tu seras capable de :

- ✅ **Intégrer** tous les concepts MLOps en un pipeline complet
- ✅ Déployer une application NLP en **production** avec monitoring
- ✅ Implémenter le **batching**, le **caching** et le **scaling**
- ✅ Livrer un **projet final** professionnel : Ticket Classifier

### Architecture complète du Jour 5

```
┌──────────────────────────────────────────────────────────────┐
│              ARCHITECTURE MLOPS COMPLÈTE — JOUR 5            │
│                                                              │
│  DATA LAYER                                                  │
│  tickets.csv ──► DVC (versioning données)                    │
│       │                                                      │
│       ▼                                                      │
│  TRAINING LAYER                                              │
│  Airflow DAG ──► extract ► preprocess ► train ► validate     │
│       │              └── MLflow tracking ──┘                 │
│       ▼                                                      │
│  MODEL REGISTRY                                              │
│  MLflow Registry (Staging → Production → Archived)           │
│       │                                                      │
│       ▼                                                      │
│  SERVING LAYER                                               │
│  FastAPI ──► /predict ──► /batch_predict ──► /health         │
│       │         └── Prometheus metrics                       │
│       ▼                                                      │
│  ORCHESTRATION                                               │
│  Kubernetes (3 replicas) + HPA (auto-scaling)                │
│       │                                                      │
│       ▼                                                      │
│  MONITORING                                                  │
│  Prometheus + Grafana + AlertManager + Drift Detection       │
└──────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 📦 INSTALLATION DES DÉPENDANCES JOUR 5
# ============================================================
import sys

!{sys.executable} -m pip install scikit-learn pandas numpy matplotlib scipy prometheus-client mlflow fastapi uvicorn pydantic --quiet

print("✅ Dépendances installées !")
print("ℹ️  Transformers (optionnel) : pip install transformers torch")

In [1]:
# ============================================================
# 📚 IMPORTS GLOBAUX
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
import time
import re
import os
import json
import pickle
import hashlib
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
from collections import Counter
from scipy.stats import ks_2samp, chi2_contingency

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print(f"✅ Tous les imports OK")
print(f"   MLflow version : {mlflow.__version__}")

✅ Tous les imports OK
   MLflow version : 3.10.1


<a id='2'></a>
## 2. 📂 Dataset — Tickets Support Client

### Contexte du Projet Final

> Tu es **ML Engineer** chez une entreprise SaaS. Le support client reçoit **500+ tickets/jour**. L'objectif est de les **classifier automatiquement** pour les router aux bonnes équipes.

### Catégories de tickets

| Catégorie | Description | Équipe |
|---|---|---|
| `BUG` | Dysfonctionnement, erreur | Dev Team |
| `FEATURE` | Demande de fonctionnalité | Product Team |
| `BILLING` | Facturation, paiement | Finance Team |
| `ACCOUNT` | Compte, accès, mot de passe | Support L1 |
| `PERFORMANCE` | Lenteur, timeout, latence | Infra Team |

In [ ]:
# ============================================================
# GÉNÉRATION DU DATASET DE TICKETS SUPPORT (200 tickets)
# ============================================================

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

TICKET_TEMPLATES = {
    'BUG': [
        "L'application crash quand je clique sur le bouton de sauvegarde",
        "Erreur 500 lors de l'exportation des données en CSV",
        "Bug critique : les données sont corrompues après la mise à jour",
        "Le module de rapport ne fonctionne plus depuis hier",
        "Impossible de se connecter, erreur de base de données",
        "L'import Excel provoque une erreur de format non attendue",
        "La fonctionnalité de recherche retourne des résultats incorrects",
        "Le tableau de bord affiche des données incohérentes",
        "Erreur lors de la synchronisation avec l'API externe",
        "Le système envoie des doublons d'emails de confirmation",
        "L'application se ferme inopinément sur mobile",
        "Les filtres de date ne fonctionnent pas correctement",
    ],
    'FEATURE': [
        "Pourriez-vous ajouter l'export en format PDF ?",
        "Il serait utile d'avoir une API REST pour automatiser les imports",
        "Suggestion : ajouter un mode sombre à l'interface",
        "Peut-on intégrer Google Calendar à votre solution ?",
        "Nous avons besoin d'un système de notifications par SMS",
        "Serait-il possible d'ajouter un module de statistiques avancées ?",
        "Demande d'intégration avec Salesforce CRM",
        "Fonctionnalité souhaitée : gestion des droits par équipe",
        "Pourriez-vous permettre le partage de liens publics ?",
        "Ajout d'un tableau Kanban pour le suivi des tâches",
        "Besoin d'une fonctionnalité de commentaires collaboratifs",
        "Intégration avec Slack pour les alertes automatiques",
    ],
    'BILLING': [
        "Je n'ai pas reçu ma facture du mois de novembre",
        "Double facturation sur ma carte de crédit ce mois-ci",
        "Comment passer à l'offre Enterprise ?",
        "Demande de remboursement pour l'abonnement non utilisé",
        "Mon paiement a été refusé mais mon compte est bien approvisionné",
        "Je souhaite changer de mode de paiement vers virement bancaire",
        "La TVA sur ma facture semble incorrecte",
        "Annulation d'abonnement et remboursement prorata",
        "Comment obtenir une facture avec notre numéro de TVA intracommunautaire ?",
        "Problème de prélèvement automatique ce mois-ci",
        "Besoin d'une facture détaillée pour ma comptabilité",
        "L'offre promotionnelle n'a pas été appliquée sur ma facture",
    ],
    'ACCOUNT': [
        "J'ai oublié mon mot de passe et le lien de réinitialisation n'arrive pas",
        "Comment ajouter un nouvel utilisateur à mon compte entreprise ?",
        "Mon compte a été suspendu sans raison apparente",
        "Je veux transférer mon compte vers une autre adresse email",
        "Impossible de me connecter avec la double authentification",
        "Comment supprimer définitivement mon compte ?",
        "Un ancien employé a toujours accès à nos données, comment révoquer ?",
        "Je ne reçois plus les emails de vérification",
        "Comment changer le nom de mon organisation dans le système ?",
        "Mon session expire trop rapidement, comment augmenter la durée ?",
        "Besoin d'activer l'authentification SSO pour mon entreprise",
        "L'administrateur est parti, comment récupérer l'accès admin ?",
    ],
    'PERFORMANCE': [
        "L'application est très lente depuis la mise à jour de la semaine dernière",
        "Les requêtes prennent plus de 30 secondes, c'est inacceptable",
        "Timeout fréquents lors du chargement des grands tableaux de bord",
        "Le temps de chargement des pages a doublé depuis hier",
        "L'export de 10 000 lignes plante le navigateur",
        "Latence élevée sur l'API REST, plus de 5 secondes par requête",
        "L'application sature la RAM de mon ordinateur",
        "Les graphiques temps réel ne se mettent plus à jour correctement",
        "Performance dégradée entre 14h et 16h chaque jour",
        "Le module de recherche full-text est extrêmement lent",
        "Les webhooks arrivent avec 10 minutes de retard",
        "CPU à 100% sur le serveur lors des traitements de nuit",
    ]
}

# Générer 200 tickets
np.random.seed(SEED)
tickets = []
ticket_id = 1000

categories = list(TICKET_TEMPLATES.keys())
# Distribution réaliste : BUG > FEATURE > BILLING > ACCOUNT > PERFORMANCE
weights = [0.35, 0.25, 0.20, 0.12, 0.08]

for _ in range(200):
    category = np.random.choice(categories, p=weights)
    templates = TICKET_TEMPLATES[category]
    base_text = random.choice(templates)

    # Varier légèrement le texte
    prefixes = ["", "Bonjour, ", "Urgent : ", "Suite à notre échange, ",
                "Depuis ce matin, ", "Problème récurrent : "]
    suffixes = ["", " Merci d'avance.", " Cordialement.",
                " Besoin d'aide rapide.", " C'est urgent !"]

    text = random.choice(prefixes) + base_text + random.choice(suffixes)

    tickets.append({
        'ticket_id' : ticket_id,
        'text'      : text,
        'category'  : category,
        'priority'  : random.choice(['LOW', 'MEDIUM', 'HIGH']),
        'created_at': (datetime(2025, 1, 1) + timedelta(days=random.randint(0, 365))).strftime('%Y-%m-%d'),
        'resolved'  : random.choice([True, False])
    })
    ticket_id += 1

df_tickets = pd.DataFrame(tickets)
df_tickets.to_csv('data/raw/tickets.csv', index=False)

print("📂 Dataset Tickets Support généré")
print("=" * 60)
print(f"  Total tickets : {len(df_tickets)}")
print(f"  Fichier       : data/raw/tickets.csv")
print()
print("  Distribution des catégories :")
cat_counts = df_tickets['category'].value_counts()
for cat, count in cat_counts.items():
    bar = '█' * int(count / 2)
    pct = count / len(df_tickets) * 100
    print(f"  {cat:12s} : {count:3d} ({pct:.0f}%)  {bar}")
print()
print(df_tickets.head(3).to_string())

<a id='3'></a>
## 3. 🧪 TP 5.1 — Pipeline Airflow NLP Complet

### Objectif
Créer un **DAG Airflow complet** avec 5 tasks pour le ticket classifier :
`extract → preprocess → train → validate → register`

In [ ]:
# ============================================================
# TP 5.1 — FONCTIONS DU PIPELINE NLP
# ============================================================

import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('ticket_pipeline')

# ---- TASK 1 : Extract ----
def task_extract():
    logger.info("[TASK 1] 📥 Extracting ticket data...")
    df = pd.read_csv('data/raw/tickets.csv')
    logger.info(f"[TASK 1] ✅ {len(df)} tickets extracted")
    df.to_csv('/tmp/tickets_extracted.csv', index=False)
    return len(df)

# ---- TASK 2 : Preprocess ----
def task_preprocess():
    logger.info("[TASK 2] 🔧 Preprocessing tickets...")
    df = pd.read_csv('/tmp/tickets_extracted.csv')

    # Nettoyage NLP
    def clean_text(text):
        text = str(text).lower()
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df['text_clean'] = df['text'].apply(clean_text)
    df = df.dropna(subset=['text_clean', 'category'])
    df = df[df['text_clean'].str.len() > 10]

    # Encodage des labels
    label_map = {'BUG': 0, 'FEATURE': 1, 'BILLING': 2, 'ACCOUNT': 3, 'PERFORMANCE': 4}
    df['label'] = df['category'].map(label_map)

    df.to_csv('/tmp/tickets_preprocessed.csv', index=False)
    logger.info(f"[TASK 2] ✅ {len(df)} tickets preprocessed")
    return len(df)

# ---- TASK 3 : Train ----
def task_train():
    logger.info("[TASK 3] 🤖 Training ticket classifier...")
    df = pd.read_csv('/tmp/tickets_preprocessed.csv')

    X = df['text_clean']
    y = df['label']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    # Pipeline TF-IDF + Logistic Regression
    model_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        )),
        ('clf', LogisticRegression(
            max_iter=500,
            random_state=SEED,
            C=1.0,
            multi_class='multinomial'
        ))
    ])

    # Tracking MLflow
    mlflow.set_tracking_uri('mlruns')
    mlflow.set_experiment('ticket_classifier')

    with mlflow.start_run(run_name='tfidf_logreg_v1') as run:
        mlflow.log_param('model', 'TF-IDF + LogisticRegression')
        mlflow.log_param('max_features', 5000)
        mlflow.log_param('ngram_range', '(1,2)')
        mlflow.log_param('train_size', len(X_train))

        model_pipeline.fit(X_train, y_train)
        y_pred = model_pipeline.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        f1  = f1_score(y_test, y_pred, average='weighted')
        prec = precision_score(y_test, y_pred, average='weighted')
        rec  = recall_score(y_test, y_pred, average='weighted')

        mlflow.log_metric('accuracy',  acc)
        mlflow.log_metric('f1_score',  f1)
        mlflow.log_metric('precision', prec)
        mlflow.log_metric('recall',    rec)

        mlflow.sklearn.log_model(
            model_pipeline, 'model',
            registered_model_name='ticket-classifier'
        )

        run_id = run.info.run_id

    # Sauvegarder localement
    model_data = {
        'model'      : model_pipeline,
        'label_map'  : {0: 'BUG', 1: 'FEATURE', 2: 'BILLING',
                        3: 'ACCOUNT', 4: 'PERFORMANCE'},
        'accuracy'   : acc,
        'f1'         : f1,
        'run_id'     : run_id,
        'version'    : '1.0'
    }
    os.makedirs('models', exist_ok=True)
    with open('models/ticket_classifier.pkl', 'wb') as f:
        pickle.dump(model_data, f)

    # Sauvegarder les données de test pour validation
    test_data = pd.DataFrame({'text': X_test, 'label': y_test})
    test_data.to_csv('/tmp/tickets_test.csv', index=False)

    logger.info(f"[TASK 3] ✅ Model trained | Accuracy={acc:.4f} | F1={f1:.4f}")
    return {'accuracy': acc, 'f1': f1, 'run_id': run_id}

# ---- TASK 4 : Validate ----
def task_validate(min_accuracy=0.75, min_f1=0.70):
    logger.info("[TASK 4] 📊 Validating model...")
    with open('models/ticket_classifier.pkl', 'rb') as f:
        model_data = pickle.load(f)

    acc = model_data['accuracy']
    f1  = model_data['f1']

    if acc < min_accuracy:
        raise ValueError(f"Accuracy {acc:.4f} < {min_accuracy}. Refusing deployment.")
    if f1 < min_f1:
        raise ValueError(f"F1 {f1:.4f} < {min_f1}. Refusing deployment.")

    logger.info(f"[TASK 4] ✅ VALIDATION PASSED | Acc={acc:.4f} | F1={f1:.4f}")
    return True

# ---- TASK 5 : Deploy ----
def task_deploy():
    logger.info("[TASK 5] 🚀 Deploying model to production...")
    with open('models/ticket_classifier.pkl', 'rb') as f:
        model_data = pickle.load(f)
    # En prod : kubectl set image deployment/ticket-api ticket-api=ticket-api:new_version
    logger.info(f"[TASK 5] ✅ Model v{model_data['version']} deployed")
    return True

print("✅ Fonctions du pipeline NLP définies (5 tasks)")

In [ ]:
# ============================================================
# EXÉCUTION DU PIPELINE COMPLET
# ============================================================

print("=" * 65)
print("  🚀 PIPELINE NLP TICKET CLASSIFIER — EXÉCUTION COMPLÈTE")
print("=" * 65)
print()

pipeline_tasks = [
    ('EXTRACT',   task_extract),
    ('PREPROCESS',task_preprocess),
    ('TRAIN',     task_train),
    ('VALIDATE',  task_validate),
    ('DEPLOY',    task_deploy),
]

pipeline_start = time.time()
task_results = {}

for task_name, task_fn in pipeline_tasks:
    t_start = time.time()
    try:
        result = task_fn()
        duration = time.time() - t_start
        task_results[task_name] = {'status': 'SUCCESS', 'duration': duration, 'result': result}
        print(f"  ✅ {task_name:12s} | {duration:.2f}s | SUCCESS")
    except Exception as e:
        duration = time.time() - t_start
        task_results[task_name] = {'status': 'FAILED', 'duration': duration, 'error': str(e)}
        print(f"  ❌ {task_name:12s} | {duration:.2f}s | FAILED: {e}")
        print("  ⚠️  Pipeline stoppé")
        break

total_time = time.time() - pipeline_start
success_n  = sum(1 for v in task_results.values() if v['status'] == 'SUCCESS')

print()
print(f"  {'='*55}")
print(f"  ⏱️  Durée totale    : {total_time:.2f}s")
print(f"  📊 Tasks réussies  : {success_n}/{len(pipeline_tasks)}")
if 'TRAIN' in task_results and task_results['TRAIN']['status'] == 'SUCCESS':
    res = task_results['TRAIN']['result']
    print(f"  🎯 Accuracy        : {res['accuracy']:.4f}")
    print(f"  🎯 F1-Score        : {res['f1']:.4f}")
print(f"  {'='*55}")

In [ ]:
# ============================================================
# ANALYSE DES RÉSULTATS + RAPPORT DE CLASSIFICATION
# ============================================================

with open('models/ticket_classifier.pkl', 'rb') as f:
    model_data = pickle.load(f)

model     = model_data['model']
label_map = model_data['label_map']

# Recharger les données de test
df_test = pd.read_csv('/tmp/tickets_test.csv')
X_test_final = df_test['text']
y_test_final = df_test['label'].astype(int)

y_pred_final = model.predict(X_test_final)

print("📊 Rapport de Classification — Ticket Classifier")
print("=" * 60)
target_names = [label_map[i] for i in sorted(label_map.keys())]
print(classification_report(y_test_final, y_pred_final,
                             target_names=target_names))

# Matrice de confusion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🎯 Ticket Classifier — Résultats', fontsize=13, fontweight='bold')

# Heatmap confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final)
im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
axes[0].set_title('Matrice de Confusion')
axes[0].set_xticks(range(len(target_names)))
axes[0].set_yticks(range(len(target_names)))
axes[0].set_xticklabels(target_names, rotation=45, ha='right')
axes[0].set_yticklabels(target_names)
axes[0].set_ylabel('Réel')
axes[0].set_xlabel('Prédit')
for i in range(len(target_names)):
    for j in range(len(target_names)):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i,j] > cm.max()/2 else 'black', fontweight='bold')

# F1 par catégorie
from sklearn.metrics import f1_score as f1_per_class
f1_scores = f1_per_class(y_test_final, y_pred_final, average=None)
bar_colors = ['#2ecc71' if f > 0.80 else '#f39c12' if f > 0.60 else '#e74c3c' for f in f1_scores]
bars = axes[1].bar(target_names, f1_scores, color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].axhline(0.75, color='red', linestyle='--', linewidth=1.5, label='Seuil minimal (0.75)')
axes[1].set_title('F1-Score par Catégorie')
axes[1].set_ylabel('F1-Score')
axes[1].set_ylim(0, 1.1)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, f1_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('ticket_classifier_results.png', dpi=100, bbox_inches='tight')
plt.show()
print("📸 Graphique sauvegardé : ticket_classifier_results.png")

In [ ]:
# ============================================================
# GÉNÉRER LE DAG AIRFLOW RÉEL
# ============================================================

os.makedirs('airflow/dags', exist_ok=True)

dag_code = '''# ticket_classification_pipeline.py
# DAG Airflow — Ticket Classifier — M1MLOP — Amadou MAIGA
#
# Placer dans : ~/airflow/dags/
# Lancer     : airflow webserver --port 8080 & airflow scheduler

from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
import pandas as pd
import numpy as np
import pickle
import re
import os
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
DATA_PATH  = \'/opt/airflow/data/raw/tickets.csv\'
MODEL_PATH = \'/opt/airflow/models/ticket_classifier.pkl\'
LABEL_MAP  = {0: \'BUG\', 1: \'FEATURE\', 2: \'BILLING\', 3: \'ACCOUNT\', 4: \'PERFORMANCE\'}
MIN_ACCURACY = 0.75

def extract_data(**context):
    df = pd.read_csv(DATA_PATH)
    df.to_csv(\'/tmp/tickets_raw.csv\', index=False)
    print(f"Extracted {len(df)} tickets")
    return len(df)

def preprocess_data(**context):
    df = pd.read_csv(\'/tmp/tickets_raw.csv\')
    df[\'text_clean\'] = df[\'text\'].str.lower().str.replace(r\'[^\\w\\s]\', \' \', regex=True)
    cat_map = {\'BUG\': 0, \'FEATURE\': 1, \'BILLING\': 2, \'ACCOUNT\': 3, \'PERFORMANCE\': 4}
    df[\'label\'] = df[\'category\'].map(cat_map)
    df = df.dropna(subset=[\'text_clean\', \'label\'])
    df.to_csv(\'/tmp/tickets_clean.csv\', index=False)
    print(f"Preprocessed {len(df)} tickets")

def train_model(**context):
    df = pd.read_csv(\'/tmp/tickets_clean.csv\')
    X_train, X_test, y_train, y_test = train_test_split(
        df[\'text_clean\'], df[\'label\'], test_size=0.2, random_state=SEED)

    pipeline = Pipeline([
        (\'tfidf\', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
        (\'clf\',   LogisticRegression(max_iter=500, random_state=SEED))
    ])
    pipeline.fit(X_train, y_train)

    acc = accuracy_score(y_test, pipeline.predict(X_test))
    f1  = f1_score(y_test, pipeline.predict(X_test), average=\'weighted\')

    mlflow.set_experiment(\'ticket_classifier\')
    with mlflow.start_run():
        mlflow.log_metric(\'accuracy\', acc)
        mlflow.log_metric(\'f1_score\', f1)
        mlflow.sklearn.log_model(pipeline, \'model\',
                                  registered_model_name=\'ticket-classifier\')

    with open(MODEL_PATH, \'wb\') as f:
        pickle.dump({\'model\': pipeline, \'accuracy\': acc, \'f1\': f1}, f)

    print(f"Model trained | Accuracy={acc:.4f} | F1={f1:.4f}")
    return acc

def validate_model(**context):
    with open(MODEL_PATH, \'rb\') as f:
        data = pickle.load(f)
    if data[\'accuracy\'] < MIN_ACCURACY:
        raise ValueError(f"Accuracy {data[\'accuracy\']} too low!")
    print(f"Validation passed: accuracy={data[\'accuracy\']}")

def deploy_model(**context):
    # En prod: kubectl rollout restart deployment/ticket-api
    print("Model deployed to production")

# ====== DÉFINIR LE DAG ======
default_args = {
    \'owner\'            : \'amadou-maiga\',
    \'retries\'          : 2,
    \'retry_delay\'      : timedelta(minutes=5),
    \'start_date\'       : datetime(2025, 12, 7),
    \'email_on_failure\'  : False,
}

dag = DAG(
    \'ticket_classification_pipeline\',
    default_args=default_args,
    description=\'Pipeline MLOps complet pour la classification de tickets support\',
    schedule_interval=\'0 2 * * *\',
    catchup=False,
    tags=[\'nlp\', \'mlops\', \'ticket-classifier\', \'m1mlop\']
)

t1 = PythonOperator(task_id=\'extract_data\',    python_callable=extract_data,    dag=dag)
t2 = PythonOperator(task_id=\'preprocess_data\',  python_callable=preprocess_data, dag=dag)
t3 = PythonOperator(task_id=\'train_model\',      python_callable=train_model,     dag=dag)
t4 = PythonOperator(task_id=\'validate_model\',   python_callable=validate_model,  dag=dag)
t5 = PythonOperator(task_id=\'deploy_model\',     python_callable=deploy_model,    dag=dag)

t1 >> t2 >> t3 >> t4 >> t5
'''

with open('airflow/dags/ticket_classification_pipeline.py', 'w') as f:
    f.write(dag_code)

print("✅ DAG Airflow généré : airflow/dags/ticket_classification_pipeline.py")
print()
print("📋 Pour utiliser ce DAG avec Airflow :")
print("   cp airflow/dags/ticket_classification_pipeline.py ~/airflow/dags/")
print("   airflow webserver --port 8080")
print("   airflow scheduler")

<a id='4'></a>
## 4. 🧪 TP 5.2 — API FastAPI avec Monitoring Prometheus

### FastAPI vs Flask

| Aspect | Flask | FastAPI |
|---|---|---|
| **Performance** | Synchrone | **Asynchrone** (2-3x plus rapide) |
| **Validation** | Manuelle | **Automatique** (Pydantic) |
| **Documentation** | Manuelle | **Auto-générée** (/docs) |
| **Type hints** | Optionnel | **Requis** |
| **Production** | Gunicorn | **Uvicorn** (ASGI) |

> ⚡ **Rapide = profitable** : Si ton API répond 100ms au lieu de 1s, tu peux servir **10x plus de requêtes** avec le même hardware.

In [ ]:
# ============================================================
# TP 5.2 — GÉNÉRER L'API FASTAPI COMPLÈTE
# ============================================================

os.makedirs('src', exist_ok=True)

fastapi_code = '''# src/api.py — API FastAPI Ticket Classifier
# M1MLOP — Amadou MAIGA — TP 5.2
#
# Lancer   : uvicorn src.api:app --host 0.0.0.0 --port 5000 --reload
# Docs     : http://localhost:5000/docs
# Métriques: http://localhost:8000/metrics

from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field
from typing import List, Optional
import pickle
import time
import hashlib
import logging
from datetime import datetime
from functools import lru_cache

from prometheus_client import (
    Counter, Gauge, Histogram,
    start_http_server, CollectorRegistry
)

# ===== CONFIGURATION =====
MODEL_PATH    = \'models/ticket_classifier.pkl\'
MODEL_VERSION = \'1.0\'
CATEGORIES    = [\'BUG\', \'FEATURE\', \'BILLING\', \'ACCOUNT\', \'PERFORMANCE\']
CATEGORY_TEAMS = {
    \'BUG\'         : \'Dev Team\',
    \'FEATURE\'     : \'Product Team\',
    \'BILLING\'     : \'Finance Team\',
    \'ACCOUNT\'     : \'Support L1\',
    \'PERFORMANCE\'  : \'Infra Team\'
}

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(\'ticket_api\')

# ===== FASTAPI APP =====
app = FastAPI(
    title="Ticket Classifier API",
    description="Classification automatique de tickets support — M1MLOP",
    version=MODEL_VERSION
)

# ===== PROMETHEUS MÉTRIQUES =====
registry = CollectorRegistry()

predictions_counter = Counter(
    \'ticket_predictions_total\',
    \'Total ticket predictions\',
    [\'category\', \'model_version\'],
    registry=registry
)
prediction_latency = Histogram(
    \'ticket_prediction_latency_seconds\',
    \'Prediction latency in seconds\',
    buckets=(0.001, 0.005, 0.01, 0.05, 0.1, 0.5),
    registry=registry
)
model_accuracy_gauge = Gauge(
    \'ticket_model_accuracy\',
    \'Current model accuracy\',
    registry=registry
)
active_requests_gauge = Gauge(
    \'ticket_active_requests\',
    \'Active requests\',
    registry=registry
)
cache_hits_counter = Counter(
    \'ticket_cache_hits_total\',
    \'Cache hits\',
    registry=registry
)

# ===== CHARGEMENT MODÈLE =====
with open(MODEL_PATH, \'rb\') as f:
    model_data = pickle.load(f)

model     = model_data[\'model\']
label_map = model_data[\'label_map\']
model_accuracy_gauge.set(model_data[\'accuracy\'])

logger.info(f"Model loaded. Accuracy={model_data[\'accuracy\']} F1={model_data[\'f1\']}")

# ===== CACHE =====
prediction_cache = {}
CACHE_MAX_SIZE = 1000

# ===== SCHÉMAS PYDANTIC =====
class TicketRequest(BaseModel):
    text: str = Field(..., min_length=5, max_length=2000,
                      description="Texte du ticket support")
    ticket_id: Optional[str] = None

class TicketResponse(BaseModel):
    ticket_id: Optional[str]
    text: str
    category: str
    team: str
    confidence: float
    all_probabilities: dict
    model_version: str
    from_cache: bool
    latency_ms: float
    timestamp: str

class BatchRequest(BaseModel):
    tickets: List[TicketRequest]

# ===== ENDPOINTS =====

@app.get(\'/health\')
async def health():
    return {
        \'status\'        : \'healthy\',
        \'model_version\'  : MODEL_VERSION,
        \'accuracy\'       : round(model_data[\'accuracy\'], 4),
        \'timestamp\'      : datetime.utcnow().isoformat()
    }

@app.get(\'/info\')
async def info():
    return {
        \'model_version\'  : MODEL_VERSION,
        \'categories\'     : CATEGORIES,
        \'team_routing\'   : CATEGORY_TEAMS,
        \'accuracy\'       : round(model_data[\'accuracy\'], 4),
        \'f1_score\'        : round(model_data[\'f1\'], 4),
        \'cache_size\'      : len(prediction_cache)
    }

@app.post(\'/predict\', response_model=TicketResponse)
async def predict(request: TicketRequest):
    active_requests_gauge.inc()
    start = time.time()

    try:
        text = request.text.strip()
        cache_key = hashlib.md5(text.encode()).hexdigest()

        # Vérifier le cache
        from_cache = cache_key in prediction_cache
        if from_cache:
            cached = prediction_cache[cache_key]
            cache_hits_counter.inc()
            result = cached
        else:
            # Prédiction
            proba    = model.predict_proba([text])[0]
            pred_idx = int(proba.argmax())
            category = label_map[pred_idx]
            conf     = float(proba[pred_idx])
            all_proba = {label_map[i]: round(float(p), 4) for i, p in enumerate(proba)}

            result = {\'category\': category, \'confidence\': conf, \'all_prob\': all_proba}

            # Mettre en cache
            if len(prediction_cache) < CACHE_MAX_SIZE:
                prediction_cache[cache_key] = result

        latency = time.time() - start

        # Métriques
        prediction_latency.observe(latency)
        predictions_counter.labels(
            category=result[\'category\'],
            model_version=MODEL_VERSION
        ).inc()

        logger.info(f"Ticket → {result[\'category\']} ({result[\'confidence\']*100:.1f}%) {latency*1000:.1f}ms")

        return TicketResponse(
            ticket_id          = request.ticket_id,
            text               = text[:200],
            category           = result[\'category\'],
            team               = CATEGORY_TEAMS[result[\'category\']],
            confidence         = round(result[\'confidence\'], 4),
            all_probabilities  = result[\'all_prob\'],
            model_version      = MODEL_VERSION,
            from_cache         = from_cache,
            latency_ms         = round(latency * 1000, 2),
            timestamp          = datetime.utcnow().isoformat()
        )

    except Exception as e:
        logger.error(f"Error: {e}")
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        active_requests_gauge.dec()

@app.post(\'/batch_predict\')
async def batch_predict(request: BatchRequest):
    start = time.time()
    results = []

    for ticket in request.tickets[:50]:  # Max 50
        text = ticket.text.strip()
        proba    = model.predict_proba([text])[0]
        pred_idx = int(proba.argmax())
        category = label_map[pred_idx]
        results.append({
            \'ticket_id\'  : ticket.ticket_id,
            \'category\'   : category,
            \'team\'       : CATEGORY_TEAMS[category],
            \'confidence\' : round(float(proba[pred_idx]), 4)
        })

    return {
        \'predictions\' : results,
        \'count\'       : len(results),
        \'latency_ms\'  : round((time.time() - start) * 1000, 2)
    }

# Exposer les métriques Prometheus
start_http_server(8000, registry=registry)

if __name__ == \'__main__\':
    import uvicorn
    uvicorn.run(app, host=\'0.0.0.0\', port=5000)
'''

with open('src/api.py', 'w') as f:
    f.write(fastapi_code)

print("✅ API FastAPI générée : src/api.py")
print()
print("📋 Endpoints :")
print("   GET  /health         → Status + accuracy")
print("   GET  /info           → Infos modèle + routing")
print("   POST /predict        → Classifier 1 ticket")
print("   POST /batch_predict  → Classifier N tickets")
print("   GET  /docs           → Swagger UI auto-généré")
print("   :8000/metrics        → Métriques Prometheus")

In [ ]:
# ============================================================
# SIMULER DES APPELS API ET GÉNÉRER LES MÉTRIQUES
# ============================================================

print("🔄 Simulation de 50 appels API au Ticket Classifier")
print("=" * 60)

test_tickets = [
    ("L'application crash quand je clique sur soumettre", 'BUG'),
    ("Pourriez-vous ajouter l'export PDF ?", 'FEATURE'),
    ("Ma facture est incorrecte ce mois-ci", 'BILLING'),
    ("Impossible de me connecter à mon compte", 'ACCOUNT'),
    ("L'application est très lente depuis hier", 'PERFORMANCE'),
    ("Erreur 500 sur la page d'accueil", 'BUG'),
    ("Je veux intégrer votre API avec Zapier", 'FEATURE'),
    ("Double prélèvement sur mon compte bancaire", 'BILLING'),
    ("Réinitialisation du mot de passe impossible", 'ACCOUNT'),
    ("Timeout fréquents lors du chargement", 'PERFORMANCE'),
]

simulation_results = []
latencies = []

np.random.seed(SEED)

for i in range(50):
    ticket_text, true_cat = random.choice(test_tickets)

    start = time.time()
    proba     = model.predict_proba([ticket_text])[0]
    pred_idx  = int(proba.argmax())
    pred_cat  = label_map[pred_idx]
    conf      = float(proba[pred_idx])
    latency   = (time.time() - start) + np.random.exponential(0.005)
    latencies.append(latency)

    simulation_results.append({
        'text'      : ticket_text[:50],
        'true_cat'  : true_cat,
        'pred_cat'  : pred_cat,
        'confidence': conf,
        'latency_ms': latency * 1000,
        'correct'   : pred_cat == true_cat
    })

sim_df  = pd.DataFrame(simulation_results)
n_correct = sim_df['correct'].sum()
acc_sim   = n_correct / len(sim_df)

print(f"  ✅ 50 prédictions effectuées")
print(f"  📊 Accuracy         : {acc_sim:.4f} ({n_correct}/50)")
print(f"  ⏱️  Latence moyenne  : {np.mean(latencies)*1000:.2f} ms")
print(f"  ⏱️  Latence P95      : {np.percentile(latencies, 95)*1000:.2f} ms")
print()
print("  Distribution des prédictions :")
for cat, count in sim_df['pred_cat'].value_counts().items():
    print(f"  {cat:12s} : {count} tickets")

<a id='5'></a>
## 5. 🧪 TP 5.3 — Optimisation : Batching & Caching

In [ ]:
# ============================================================
# TP 5.3 — BATCHING ET CACHING
# ============================================================

print("⚡ Optimisation — Batching vs Individual vs Caching")
print("=" * 60)

test_texts_perf = [
    "L'application crash quand j'exporte les données",
    "Demande d'ajout d'une fonctionnalité de rapport PDF",
    "Ma facture du mois de novembre est incorrecte",
    "Impossible de me connecter avec mon mot de passe",
    "Le temps de chargement est très lent depuis hier",
] * 20  # 100 tickets

# ---- Méthode 1 : Individuelle (une par une) ----
start = time.time()
results_individual = []
for text in test_texts_perf:
    pred = model.predict([text])[0]
    results_individual.append(pred)
time_individual = time.time() - start

# ---- Méthode 2 : Batch (toutes en une fois) ----
start = time.time()
results_batch = model.predict(test_texts_perf)
time_batch = time.time() - start

# ---- Méthode 3 : Avec cache (LRU) ----
cache = {}
cache_hits = 0

start = time.time()
results_cached = []
for text in test_texts_perf:
    key = hashlib.md5(text.encode()).hexdigest()
    if key in cache:
        results_cached.append(cache[key])
        cache_hits += 1
    else:
        pred = model.predict([text])[0]
        cache[key] = pred
        results_cached.append(pred)
time_cached = time.time() - start

n = len(test_texts_perf)

print(f"  {n} prédictions — Comparaison des méthodes :")
print(f"  {'─'*55}")
print(f"  {'Méthode':25s} {'Temps':10s} {'ms/req':10s} {'Speedup'}")
print(f"  {'─'*55}")
print(f"  {'Individuelle':25s} {time_individual:.4f}s  {time_individual/n*1000:.2f}ms    1.0x (référence)")
print(f"  {'Batch':25s} {time_batch:.4f}s  {time_batch/n*1000:.2f}ms    {time_individual/time_batch:.1f}x plus rapide")
print(f"  {'Cache (après 1er run)':25s} {time_cached:.4f}s  {time_cached/n*1000:.2f}ms    {time_individual/time_cached:.1f}x plus rapide")
print(f"  {'─'*55}")
print(f"  Cache hits : {cache_hits}/{n} ({cache_hits/n*100:.0f}%) — {len(cache)} entrées uniques")
print()
print(f"  💡 En production (GPU + grands modèles) :")
print(f"     Batch processing → 10-25x plus rapide")
print(f"     Caching Redis    → quasi-instantané pour les requêtes répétées")

In [ ]:
# ============================================================
# VISUALISATION OPTIMISATION
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('⚡ Optimisation API — Batching & Caching', fontsize=13, fontweight='bold')

# ---- Comparaison temps ----
methods = ['Individuelle\n(1 par 1)', 'Batch\n(tout d\'un coup)', 'Avec Cache\n(LRU)']
times   = [time_individual, time_batch, time_cached]
colors  = ['#e74c3c', '#3498db', '#2ecc71']

bars = axes[0].bar(methods, [t*1000 for t in times], color=colors, alpha=0.85, edgecolor='white')
axes[0].set_title(f'Temps total pour {n} prédictions')
axes[0].set_ylabel('Temps (ms)')
axes[0].grid(axis='y', alpha=0.3)
for bar, t in zip(bars, times):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{t*1000:.1f}ms', ha='center', fontweight='bold', fontsize=10)

# ---- Speedup ----
speedups = [1.0, time_individual/time_batch, time_individual/time_cached]
bars2 = axes[1].bar(methods, speedups, color=colors, alpha=0.85, edgecolor='white')
axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Accélération (Speedup) par rapport à individuelle')
axes[1].set_ylabel('Speedup (x fois plus rapide)')
axes[1].grid(axis='y', alpha=0.3)
for bar, sp in zip(bars2, speedups):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{sp:.1f}x', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('optimization_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("📸 Graphique sauvegardé : optimization_comparison.png")

<a id='6'></a>
## 6. 🧪 TP 5.4 — Kubernetes : HPA + Rolling Update

In [ ]:
# ============================================================
# TP 5.4 — MANIFESTS KUBERNETES COMPLETS
# ============================================================

os.makedirs('kubernetes', exist_ok=True)

# ---- deployment.yaml ----
deployment_yaml = '''# kubernetes/deployment.yaml
# Ticket Classifier — M1MLOP — Amadou MAIGA
# Déployer : kubectl apply -f kubernetes/

apiVersion: apps/v1
kind: Deployment
metadata:
  name: ticket-classifier
  labels:
    app: ticket-classifier
    version: "1.0"
    module: m1mlop
spec:
  replicas: 3
  strategy:
    type: RollingUpdate          # Zero-downtime deployment
    rollingUpdate:
      maxSurge: 1                # +1 pod pendant le déploiement
      maxUnavailable: 0          # 0 pod down pendant le déploiement
  selector:
    matchLabels:
      app: ticket-classifier
  template:
    metadata:
      labels:
        app: ticket-classifier
    spec:
      containers:
      - name: ticket-api
        image: ticket-classifier:1.0
        ports:
        - containerPort: 5000
          name: api
        - containerPort: 8000
          name: metrics
        resources:
          requests:
            memory: "256Mi"
            cpu: "250m"
          limits:
            memory: "512Mi"
            cpu: "500m"
        env:
        - name: MODEL_VERSION
          value: "1.0"
        - name: LOG_LEVEL
          value: "INFO"
        livenessProbe:
          httpGet:
            path: /health
            port: 5000
          initialDelaySeconds: 30
          periodSeconds: 10
          failureThreshold: 3
        readinessProbe:
          httpGet:
            path: /health
            port: 5000
          initialDelaySeconds: 10
          periodSeconds: 5
'''

service_yaml = '''# kubernetes/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: ticket-classifier-service
spec:
  type: LoadBalancer
  selector:
    app: ticket-classifier
  ports:
  - name: api
    port: 80
    targetPort: 5000
  - name: metrics
    port: 8000
    targetPort: 8000
---
# ServiceMonitor pour Prometheus Operator
apiVersion: monitoring.coreos.com/v1
kind: ServiceMonitor
metadata:
  name: ticket-classifier-monitor
spec:
  selector:
    matchLabels:
      app: ticket-classifier
  endpoints:
  - port: metrics
    interval: 30s
    path: /metrics
'''

hpa_yaml = '''# kubernetes/hpa.yaml — Auto Horizontal Scaling
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: ticket-classifier-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: ticket-classifier
  minReplicas: 3
  maxReplicas: 15
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70     # Scale si CPU > 70%
  - type: Resource
    resource:
      name: memory
      target:
        type: Utilization
        averageUtilization: 80     # Scale si RAM > 80%
  behavior:
    scaleUp:
      stabilizationWindowSeconds: 60   # Attendre 60s avant de scale up
      policies:
      - type: Pods
        value: 3
        periodSeconds: 60              # Max +3 pods par minute
    scaleDown:
      stabilizationWindowSeconds: 300  # Attendre 5min avant de scale down
'''

# Dockerfile projet final
dockerfile = '''# Dockerfile — Ticket Classifier Production
# M1MLOP — Amadou MAIGA
FROM python:3.10-slim

WORKDIR /app

RUN apt-get update && apt-get install -y curl && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY src/ ./src/
COPY models/ ./models/
COPY data/ ./data/

EXPOSE 5000 8000

HEALTHCHECK --interval=30s --timeout=10s --start-period=30s \\\
  CMD curl -f http://localhost:5000/health || exit 1

CMD ["uvicorn", "src.api:app", "--host", "0.0.0.0", "--port", "5000"]
'''

# docker-compose.yml
docker_compose = '''# docker-compose.yml — Dev environment
# Lancer : docker-compose up
version: \'3.8\'
services:
  ticket-api:
    build: .
    ports:
      - "5000:5000"
      - "8000:8000"
    volumes:
      - ./models:/app/models
      - ./data:/app/data
    environment:
      - MODEL_VERSION=1.0
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:5000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

  prometheus:
    image: prom/prometheus:latest
    ports:
      - "9090:9090"
    volumes:
      - ./monitoring/prometheus.yml:/etc/prometheus/prometheus.yml

  grafana:
    image: grafana/grafana:latest
    ports:
      - "3000:3000"
    environment:
      - GF_SECURITY_ADMIN_PASSWORD=admin
'''

with open('kubernetes/deployment.yaml', 'w') as f:
    f.write(deployment_yaml)
with open('kubernetes/service.yaml', 'w') as f:
    f.write(service_yaml)
with open('kubernetes/hpa.yaml', 'w') as f:
    f.write(hpa_yaml)
with open('Dockerfile', 'w') as f:
    f.write(dockerfile)
with open('docker-compose.yml', 'w') as f:
    f.write(docker_compose)

print("✅ Fichiers Kubernetes générés :")
for fname in ['kubernetes/deployment.yaml', 'kubernetes/service.yaml',
              'kubernetes/hpa.yaml', 'Dockerfile', 'docker-compose.yml']:
    size = os.path.getsize(fname)
    print(f"   {fname:35s} {size:>5} bytes")

print()
print("📋 Commandes kubectl :")
print("   kubectl apply -f kubernetes/")
print("   kubectl get pods")
print("   kubectl get hpa")
print("   kubectl rollout status deployment/ticket-classifier")
print("   kubectl scale deployment ticket-classifier --replicas=5")

<a id='7'></a>
## 7. 🎯 PROJET FINAL — Ticket Classifier System

### Critères d'évaluation

| Critère | Points | Status |
|---|---|---|
| Modèle ML (Accuracy ≥ 80%, F1 ≥ 0.75) | 20 | ✅ |
| Pipeline Airflow (4+ tasks) | 20 | ✅ |
| API FastAPI (endpoints + validation) | 15 | ✅ |
| Monitoring (Prometheus + logs) | 15 | ✅ |
| Déploiement (Dockerfile + K8s) | 15 | ✅ |
| Code Quality (tests + docs) | 10 | ✅ |
| Présentation | 5 | ✅ |
| **TOTAL** | **100** | |

In [ ]:
# ============================================================
# STRUCTURE COMPLÈTE DU PROJET FINAL
# ============================================================

print("📁 Structure du Projet Final — ticket-classifier")
print("=" * 55)
print('''
ticket-classifier/
├── 📂 data/
│   ├── raw/
│   │   └── tickets.csv            ← 200 tickets labelisés
│   └── processed/
│       └── train_test_split.csv
│
├── 📂 models/
│   └── ticket_classifier.pkl      ← TF-IDF + LogReg entraîné
│
├── 📂 src/
│   ├── __init__.py
│   ├── api.py                     ← FastAPI (4 endpoints)
│   ├── data_processing.py
│   ├── model_training.py
│   └── drift_detection.py
│
├── 📂 airflow/
│   └── dags/
│       └── ticket_classification_pipeline.py  ← DAG 5 tasks
│
├── 📂 kubernetes/
│   ├── deployment.yaml            ← 3 replicas + rolling update
│   ├── service.yaml               ← LoadBalancer
│   └── hpa.yaml                   ← Auto-scaling CPU/RAM
│
├── 📂 tests/
│   ├── test_model.py
│   ├── test_api.py
│   └── test_pipeline.py
│
├── 📂 monitoring/
│   └── prometheus.yml
│
├── 🐳 Dockerfile
├── 🐳 docker-compose.yml
├── 📋 requirements.txt
└── 📖 README.md
''')

<a id='8'></a>
## 8. 🧪 Tests Automatisés du Projet Final

In [ ]:
# ============================================================
# GÉNÉRER LES TESTS PYTEST DU PROJET FINAL
# ============================================================

os.makedirs('tests', exist_ok=True)

test_model_code = '''# tests/test_model.py — Tests du Ticket Classifier
# M1MLOP — Amadou MAIGA
# Lancer : pytest tests/ -v

import pytest
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

MODEL_PATH = \'models/ticket_classifier.pkl\'
CATEGORIES = [\'BUG\', \'FEATURE\', \'BILLING\', \'ACCOUNT\', \'PERFORMANCE\']

@pytest.fixture(scope=\'module\')
def model_data():
    with open(MODEL_PATH, \'rb\') as f:
        return pickle.load(f)

class TestModelQuality:

    def test_accuracy_above_threshold(self, model_data):
        """Le modèle doit atteindre au moins 75% d\'accuracy."""
        assert model_data[\'accuracy\'] >= 0.75, \\\
            f"Accuracy {model_data[\'accuracy\']} < 0.75"

    def test_f1_above_threshold(self, model_data):
        """Le F1-score doit être >= 0.70."""
        assert model_data[\'f1\'] >= 0.70, \\\
            f"F1 {model_data[\'f1\']} < 0.70"

    def test_model_predicts_all_categories(self, model_data):
        """Le modèle doit pouvoir prédire toutes les catégories."""
        model = model_data[\'model\']
        test_texts = [
            "Application crash erreur 500",
            "Demande ajout fonctionnalité export PDF",
            "Ma facture est incorrecte",
            "Impossible de me connecter",
            "Application très lente depuis hier"
        ]
        preds = model.predict(test_texts)
        assert len(set(preds)) == 5, \\\
            f"Model predicts only {len(set(preds))} categories"

class TestModelRobustness:

    def test_single_word_input(self, model_data):
        """Le modèle gère un texte très court."""
        model = model_data[\'model\']
        pred  = model.predict([\'bug\'])
        assert len(pred) == 1

    def test_long_text_input(self, model_data):
        """Le modèle gère un texte très long."""
        model = model_data[\'model\']
        long_text = \'Application crash \' * 100
        pred = model.predict([long_text])
        assert len(pred) == 1

    def test_output_is_valid_category(self, model_data):
        """Toute prédiction doit être dans les catégories valides."""
        model = model_data[\'model\']
        texts = [\'test ticket 1\', \'test ticket 2\', \'test ticket 3\']
        preds = model.predict(texts)
        valid_labels = {0, 1, 2, 3, 4}
        assert set(preds).issubset(valid_labels)

    def test_predict_proba_sums_to_one(self, model_data):
        """Les probabilités doivent sommer à 1."""
        model = model_data[\'model\']
        proba = model.predict_proba([\'test ticket\'])[0]
        assert abs(sum(proba) - 1.0) < 1e-6

    def test_batch_prediction(self, model_data):
        """La prédiction en batch retourne le bon nombre de résultats."""
        model = model_data[\'model\']
        texts = [f\'ticket test {i}\' for i in range(20)]
        preds = model.predict(texts)
        assert len(preds) == 20
'''

test_pipeline_code = '''# tests/test_pipeline.py — Tests du Pipeline
# M1MLOP — Amadou MAIGA

import pytest
import pandas as pd
import re
import os

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r\'[^\\w\\s]\', \' \', text)
    return re.sub(r\'\\s+\', \' \', text).strip()

class TestPreprocessing:

    def test_lowercase(self):
        assert clean_text(\'URGENT BUG\') == \'urgent bug\'

    def test_remove_punctuation(self):
        result = clean_text(\'App crash!!! Error 500.\')  
        assert \'!\' not in result
        assert \'.\'  not in result

    def test_empty_string(self):
        result = clean_text(\'\')  
        assert isinstance(result, str)

    def test_preserves_words(self):
        result = clean_text(\'application crash erreur\')
        assert \'application\' in result
        assert \'crash\' in result

class TestDataset:

    def test_dataset_exists(self):
        assert os.path.exists(\'data/raw/tickets.csv\'), \'Dataset not found\'

    def test_dataset_has_required_columns(self):
        df = pd.read_csv(\'data/raw/tickets.csv\')
        required = {\'ticket_id\', \'text\', \'category\'}
        assert required.issubset(set(df.columns))

    def test_dataset_no_empty_texts(self):
        df = pd.read_csv(\'data/raw/tickets.csv\')
        assert df[\'text\'].notna().all()
        assert (df[\'text\'].str.len() > 5).all()

    def test_dataset_valid_categories(self):
        df   = pd.read_csv(\'data/raw/tickets.csv\')
        valid = {\'BUG\', \'FEATURE\', \'BILLING\', \'ACCOUNT\', \'PERFORMANCE\'}
        assert set(df[\'category\']).issubset(valid)

    def test_dataset_size(self):
        df = pd.read_csv(\'data/raw/tickets.csv\')
        assert len(df) >= 100, f\'Dataset too small: {len(df)} tickets\'
'''

with open('tests/test_model.py', 'w') as f:
    f.write(test_model_code)
with open('tests/test_pipeline.py', 'w') as f:
    f.write(test_pipeline_code)
with open('tests/__init__.py', 'w') as f:
    f.write('')

print("✅ Tests pytest générés :")
print("   tests/test_model.py    (TestModelQuality + TestModelRobustness)")
print("   tests/test_pipeline.py (TestPreprocessing + TestDataset)")
print()
print("📋 Pour lancer les tests :")
print("   pytest tests/ -v --tb=short")

In [ ]:
# ============================================================
# EXÉCUTER LES TESTS DIRECTEMENT DANS LE NOTEBOOK
# ============================================================

print("🧪 Exécution des tests — Projet Final")
print("=" * 60)

with open('models/ticket_classifier.pkl', 'rb') as f:
    md = pickle.load(f)

m = md['model']
lm = md['label_map']

test_suite = [
    # (nom_test, condition, description)
    ("accuracy >= 0.75",          md['accuracy'] >= 0.75,
     f"Accuracy={md['accuracy']:.4f}"),
    ("f1 >= 0.70",                 md['f1'] >= 0.70,
     f"F1={md['f1']:.4f}"),
    ("5 catégories disponibles",   len(set(m.predict([
        'crash erreur 500', 'ajouter fonctionnalité export',
        'facture incorrecte', 'impossible connexion',
        'application lente'
    ]))) == 5,                     "Toutes les classes prédites"),
    ("proba somme à 1",            abs(sum(m.predict_proba(['test'])[0]) - 1.0) < 1e-5,
     "Cohérence des probabilités"),
    ("batch de 20",               len(m.predict([f'ticket {i}' for i in range(20)])) == 20,
     "20 prédictions en batch"),
    ("texte court",               len(m.predict(['bug'])) == 1,
     "1 mot fonctionne"),
    ("dataset existe",            os.path.exists('data/raw/tickets.csv'),
     "data/raw/tickets.csv"),
    ("modèle sauvegardé",         os.path.exists('models/ticket_classifier.pkl'),
     "models/ticket_classifier.pkl"),
    ("DAG Airflow généré",        os.path.exists('airflow/dags/ticket_classification_pipeline.py'),
     "DAG Python"),
    ("API FastAPI générée",       os.path.exists('src/api.py'),
     "src/api.py"),
    ("Dockerfile créé",           os.path.exists('Dockerfile'),
     "Dockerfile"),
    ("K8s deployment.yaml",       os.path.exists('kubernetes/deployment.yaml'),
     "deployment.yaml"),
    ("Tests pytest créés",        os.path.exists('tests/test_model.py'),
     "test_model.py"),
]

passed = 0
failed = 0
for name, condition, detail in test_suite:
    if condition:
        print(f"  ✅ {name:40s} ({detail})")
        passed += 1
    else:
        print(f"  ❌ {name:40s} ({detail})")
        failed += 1

print()
print("=" * 60)
print(f"  📊 {passed}/{len(test_suite)} tests PASSED")
if failed == 0:
    print("  🏆 Tous les tests sont VERTS !")
else:
    print(f"  ⚠️  {failed} test(s) échoué(s)")
print("=" * 60)

<a id='9'></a>
## 9. 📝 Résumé de la Formation & Bilan Final

### 🎓 Récapitulatif des 5 jours

| Jour | Thème | Outils | TPs |
|---|---|---|---|
| **Jour 1** | Fondements MLOps | MLflow | Tracking, Registry, Rollback |
| **Jour 2** | Pipelines & Orchestration | Airflow, Docker, K8s | DAGs, CI/CD, API Flask |
| **Jour 3** | Monitoring & Gouvernance | Prometheus, SHAP | Drift, A/B Test, Fairness |
| **Jour 4** | NLP & Transformers | BERT, HuggingFace | Fine-tuning, NER, QA |
| **Jour 5** | Intégration & Projet Final | FastAPI, K8s HPA | Ticket Classifier complet |

### ✅ Compétences acquises

```
MLOps Stack :
  ✅ MLflow     — tracking, registry, versioning, rollback
  ✅ Airflow    — DAGs, scheduling, parallélisation, retries
  ✅ Docker     — conteneurisation, Dockerfile, docker-compose
  ✅ Kubernetes — Deployment, Service, HPA, rolling update
  ✅ Prometheus — Counter, Gauge, Histogram, alertes

NLP Stack :
  ✅ Preprocessing  — tokenisation, TF-IDF, stopwords
  ✅ Transformers   — BERT, DistilBERT, fine-tuning
  ✅ Applications   — sentiment analysis, NER, QA
  ✅ Production     — API FastAPI, batch predict, caching

Gouvernance :
  ✅ Drift Detection — KS test, concept drift
  ✅ Explicabilité  — SHAP, feature importance
  ✅ Fairness       — détection de biais, RGPD
  ✅ A/B Testing    — Chi-square, significativité statistique
```

### 🚀 Prochaines étapes

- **M1DISTR (SEM12)** : Architectures BigData distribuées avec intégration MLOps
- **M1INNO (SEM13)** : Projets innovants appliquant MLOps et NLP
- **Certifications** : AWS ML Specialty, Google ML Engineer, Databricks MLflow
- **Portfolio** : Ce repo GitHub est la base de votre portfolio MLOps !

In [ ]:
# ============================================================
# ✅ BILAN FINAL — FORMATION M1MLOP COMPLÈTE
# ============================================================

print("=" * 65)
print("   🎓 BILAN FINAL — M1MLOP MLOps & NLP — Amadou MAIGA")
print("=" * 65)
print()

all_deliverables = [
    ("Jour 1", "MLflow Tracking & Registry",                True),
    ("Jour 2", "Pipeline Airflow + Docker + Kubernetes",    True),
    ("Jour 3", "Monitoring + Drift + A/B + Gouvernance",    True),
    ("Jour 4", "NLP + BERT Fine-tuning + NER + API",        True),
    ("Jour 5", "Ticket Classifier complet",                 True),
    ("Projet", "data/raw/tickets.csv (200 tickets)",
               os.path.exists('data/raw/tickets.csv')),
    ("Projet", "models/ticket_classifier.pkl",
               os.path.exists('models/ticket_classifier.pkl')),
    ("Projet", "src/api.py (FastAPI)",
               os.path.exists('src/api.py')),
    ("Projet", "airflow/dags/ticket_pipeline.py",
               os.path.exists('airflow/dags/ticket_classification_pipeline.py')),
    ("Projet", "kubernetes/ (deployment + service + hpa)",
               all(os.path.exists(f'kubernetes/{f}.yaml')
                   for f in ['deployment','service','hpa'])),
    ("Projet", "Dockerfile + docker-compose.yml",
               os.path.exists('Dockerfile')),
    ("Projet", "tests/ (test_model + test_pipeline)",
               os.path.exists('tests/test_model.py')),
]

for category, label, ok in all_deliverables:
    icon = "✅" if ok else "❌"
    print(f"  {icon} [{category:6s}] {label}")

print()
print("=" * 65)
print(f"  🎯 Modèle : Accuracy = {md['accuracy']:.4f} | F1 = {md['f1']:.4f}")
print("=" * 65)
print()
print("  📌 Git commit final :")
print("  git add .")
print('  git commit -m "🎓 Formation M1MLOP complète - 5 jours - Amadou MAIGA"')
print("  git push")
print()
print("  🌐 GitHub repo : https://github.com/Amadou-M/mlops-nlp-master1")
print("=" * 65)
print()
print("         🏆 FÉLICITATIONS — FORMATION M1MLOP TERMINÉE !")
print("=" * 65)